# HotelIQ – Hotel Business Intelligence & Cancellation Risk Analytics

**Author:** Ishita Ghosh (AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026)  
**Project Goal:** End-to-End Business Intelligence, Data Quality Pipeline, Statistical Exploratory Data Analysis, Cancellation Driver Analysis, and Machine Learning Risk Modeling on Hotel Booking Data.

---

## Phase 1: Data Loading & Dataset Audit

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

# Load raw dataset
df_raw = pd.read_csv('hotel_bookings.csv')
print('Raw Dataset Shape:', df_raw.shape)
df_raw.head()

### Data Quality Inspection (Missing Values, Duplicates & Data Types)

In [ ]:
print('=== MISSING VALUES ===')
nulls = df_raw.isnull().sum()[df_raw.isnull().sum() > 0]
print(nulls)

print('\n=== DUPLICATE ROWS ===')
print('Duplicates Count:', df_raw.duplicated().sum(), f'({round(df_raw.duplicated().sum()/len(df_raw)*100, 2)}%)')

print('\n=== DATA TYPES ===')
print(df_raw.dtypes.value_counts())

## Data Cleaning Pipeline
1. Impute missing values (`children` -> 0, `agent` -> 0, `company` -> 0, `country` -> 'Unknown').
2. Standardize `meal` categories ('Undefined' -> 'SC').
3. Remove invalid zero-guest records (`adults + children + babies == 0`) and invalid market/channel records.
4. Correct ADR anomalies (remove `adr < 0` and extreme outlier `adr = 5400`).
5. Date parsing and string formatting.

In [ ]:
df_clean = df_raw.copy()

# 1. Imputation
df_clean['children'] = df_clean['children'].fillna(0).astype(int)
df_clean['country'] = df_clean['country'].fillna('Unknown')
df_clean['agent'] = df_clean['agent'].fillna(0).astype(int)
df_clean['company'] = df_clean['company'].fillna(0).astype(int)

# 2. Remap meal
df_clean['meal'] = df_clean['meal'].replace('Undefined', 'SC')

# 3. Remove invalid rows
valid_market = df_clean['market_segment'] != 'Undefined'
valid_dist = df_clean['distribution_channel'] != 'Undefined'
valid_guests = (df_clean['adults'] + df_clean['children'] + df_clean['babies']) > 0
valid_adr = (df_clean['adr'] >= 0) & (df_clean['adr'] < 5000)

df_clean = df_clean[valid_market & valid_dist & valid_guests & valid_adr].copy()

# 4. Date parsing
months_map = {'January':1, 'February':2, 'March':3, 'April':4, 'May':5, 'June':6, 'July':7, 'August':8, 'September':9, 'October':10, 'November':11, 'December':12}
df_clean['arrival_month_num'] = df_clean['arrival_date_month'].map(months_map)
df_clean['arrival_date'] = pd.to_datetime(
    df_clean['arrival_date_year'].astype(str) + '-' +
    df_clean['arrival_month_num'].astype(str).str.zfill(2) + '-' +
    df_clean['arrival_date_day_of_month'].astype(str).str.zfill(2)
)
df_clean['reservation_status_date'] = pd.to_datetime(df_clean['reservation_status_date'])

print('Cleaned Shape:', df_clean.shape)
print('Remaining Nulls:', df_clean.isnull().sum().sum())

## Phase 2: Feature Engineering

In [ ]:
df_clean['total_stay_nights'] = df_clean['stays_in_weekend_nights'] + df_clean['stays_in_week_nights']
df_clean['total_guests'] = df_clean['adults'] + df_clean['children'] + df_clean['babies']
df_clean['estimated_booking_value'] = df_clean['adr'] * df_clean['total_stay_nights']
df_clean['is_room_changed'] = (df_clean['reserved_room_type'] != df_clean['assigned_room_type']).astype(int)

bins = [-1, 0, 7, 30, 90, 180, 365, 1000]
labels = ['Same Day', '1-7 Days', '8-30 Days', '31-90 Days', '91-180 Days', '181-365 Days', '>365 Days']
df_clean['lead_time_group'] = pd.cut(df_clean['lead_time'], bins=bins, labels=labels)

print('Engineered Features Created Successfully!')
df_clean[['total_stay_nights', 'total_guests', 'estimated_booking_value', 'is_room_changed', 'lead_time_group']].head()

## Phase 3: Exploratory Data Analysis & Driver Analysis

In [ ]:
# Overall KPIs
tot_b = len(df_clean)
tot_c = df_clean['is_canceled'].sum()
c_rate = tot_c / tot_b * 100
avg_adr = df_clean['adr'].mean()
tot_val = df_clean['estimated_booking_value'].sum()
lost_val = df_clean[df_clean['is_canceled'] == 1]['estimated_booking_value'].sum()

print(f'Total Bookings: {tot_b:,}')
print(f'Cancelled Bookings: {tot_c:,} ({c_rate:.2f}%)')
print(f'Average ADR: ${avg_adr:.2f}')
print(f'Total Estimated Booking Value: ${tot_val:,.2f}')
print(f'Lost Estimated Booking Value: ${lost_val:,.2f} ({lost_val/tot_val*100:.2f}%)')

In [ ]:
# Hotel Type & Deposit Type Breakdown Plots
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=df_clean, x='hotel', hue='is_canceled', palette='coolwarm', ax=ax[0])
ax[0].set_title('Bookings by Hotel Type & Cancellation Status')

sns.barplot(data=df_clean, x='deposit_type', y='is_canceled', palette='viridis', ax=ax[1])
ax[1].set_title('Cancellation Rate by Deposit Type')
plt.tight_layout()
plt.show()

In [ ]:
# Lead Time & Room Change Driver Analysis
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=df_clean, x='lead_time_group', y='is_canceled', palette='magma', ax=ax[0])
ax[0].set_title('Cancellation Rate by Lead Time Bin')
ax[0].tick_params(axis='x', rotation=30)

sns.barplot(data=df_clean, x='is_room_changed', y='is_canceled', palette='Set2', ax=ax[1])
ax[1].set_title('Cancellation Rate: Reserved vs Assigned Room Change')
ax[1].set_xticklabels(['Room Unchanged', 'Room Changed/Upgraded'])
plt.tight_layout()
plt.show()

## Phase 4: Machine Learning — Cancellation Risk Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Use deduplicated dataset for ML model training to prevent train-test data leakage
df_dedup = df_clean.drop_duplicates().copy()

# Feature selection (excluding target leakage)
drop_cols = ['is_canceled', 'reservation_status', 'reservation_status_date', 'arrival_date', 'arrival_month_num']
feature_cols = [c for c in df_dedup.columns if c not in drop_cols]

X = df_dedup[feature_cols]
y = df_dedup['is_canceled']

cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

# 1. Logistic Regression Model
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])
lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)
y_prob_lr = lr_pipeline.predict_proba(X_test)[:, 1]

# 2. Random Forest Classifier Model
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1))
])
rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)
y_prob_rf = rf_pipeline.predict_proba(X_test)[:, 1]

print('=== LOGISTIC REGRESSION REPORT ===')
print(classification_report(y_test, y_pred_lr))
print('ROC-AUC:', round(roc_auc_score(y_test, y_prob_lr), 4))

print('\n=== RANDOM FOREST REPORT ===')
print(classification_report(y_test, y_pred_rf))
print('ROC-AUC:', round(roc_auc_score(y_test, y_prob_rf), 4))

In [ ]:
# Confusion Matrix & Feature Importance Plots
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Blues', ax=ax[0])
ax[0].set_title('Random Forest Confusion Matrix')
ax[0].set_xlabel('Predicted Label')
ax[0].set_ylabel('True Label')

# Feature Importance
encoder = rf_pipeline.named_steps['preprocessor'].named_transformers_['cat']
cat_feature_names = encoder.get_feature_names_out(cat_cols).tolist()
all_feature_names = num_cols + cat_feature_names
importances = rf_pipeline.named_steps['classifier'].feature_importances_

feat_imp = pd.Series(importances, index=all_feature_names).sort_values(ascending=False).head(10)
feat_imp.plot(kind='barh', ax=ax[1], color='teal')
ax[1].set_title('Top 10 Predictors of Cancellation Risk')
ax[1].invert_yaxis()
plt.tight_layout()
plt.show()

## Phase 5: Business Insights & Strategic Recommendations

### Finding 1: Lead Time Risk
- **FACT:** Bookings with lead times >180 days have a cancellation rate of 55.46% (rising to 67.65% for >365 days), compared to only 10.96% for 1-7 days.
- **INSIGHT:** Long lead times create speculative bookings where guests lock in options without firm commitment.
- **RISK / OPPORTUNITY:** High inventory holding costs and unfulfilled room allocation.
- **POSSIBLE BUSINESS ACTION:** Implement staggered non-refundable deposits and automated re-confirmation touchpoints at 90, 60, and 30 days prior to arrival.

### Finding 2: Non-Refundable Deposit Paradox
- **FACT:** Non-refundable deposit bookings show a 99.36% cancellation rate (14,493 out of 14,586 canceled).
- **INSIGHT:** Tour operators/agencies block large room blocks with non-refundable tags far in advance and relinquish unallocated blocks when group demand fails to materialize.
- **RISK / OPPORTUNITY:** Phantom demand distorting revenue forecasts.
- **POSSIBLE BUSINESS ACTION:** Restructure agency distribution contracts; require pre-payment guarantees or release deadlines for wholesale group blocks.